In [1]:
import os
import time
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from openpyxl.styles import Alignment



# 配置 Chrome 浏览器选项
options = Options()

# 不保存登录状态
options.add_argument("--incognito")

#改数字来切换账号
account = "account2"

options.add_argument(
    rf"--user-data-dir=C:\Users\qieziclub1660ti\AppData\Local\Google\Chrome\Wechat_{account}"
)

options.add_argument("--start-maximized")

options.add_argument("--disable-gpu")

options.add_argument("--no-sandbox")

options.add_argument("--disable-dev-shm-usage")


options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)


options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

# 创建 Chrome WebDriver
webdriver_path = r"C:\WebDriver\chromedriver.exe"

driver = webdriver.Chrome(
    service=Service(webdriver_path),
    options=options
)




# =========================
# 打开公众号后台
# =========================

url = "https://mp.weixin.qq.com/"


driver.get(url)



wait = WebDriverWait(driver, 300)



# =========================
# 登录等待
# =========================

try:

    print("请登录微信公众号...")


    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//span[text()='内容管理']"
            )
        )
    )


    print("登录成功！")


except Exception as e:

    print(
        "登录失败:",
        e
    )

    driver.quit()

    exit()



# =========================
# 进入发表记录
# =========================

try:


    driver.find_element(
        By.XPATH,
        "//span[text()='内容管理']"
    ).click()



    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//span[text()='发表记录']"
            )
        )
    ).click()



    print(
        "进入发表记录页面"
    )


except Exception as e:

    print(
        "进入页面失败:",
        e
    )

    driver.quit()

    exit()



# =========================
# 数据提取函数
# =========================

def clean_number(text):

    if not text:
        return "0"


    return (
        text
        .replace(",","")
        .strip()
    )



def extract_page_data():


    page_data = []


    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )


    articles = soup.find_all(
        "div",
        class_="weui-desktop-mass-appmsg__bd"
    )


    for article in articles:


        try:


            title_tag = article.find(
                "a",
                class_="weui-desktop-mass-appmsg__title"
            )


            title = (
                title_tag.find("span").text.strip()
                if title_tag
                else "N/A"
            )



            def get_data(class_name):

                item = article.find(
                    "div",
                    class_=class_name
                )


                if item:

                    span = item.find(
                        "span",
                        class_="weui-desktop-mass-media__data__inner"
                    )

                    if span:

                        return clean_number(
                            span.text
                        )


                return "0"



            read_count = get_data(
                "weui-desktop-mass-media__data appmsg-view"
            )


            likes = get_data(
                "weui-desktop-mass-media__data appmsg-like"
            )


            shares = get_data(
                "weui-desktop-mass-media__data appmsg-share"
            )


            on_view = get_data(
                "weui-desktop-mass-media__data appmsg-haokan"
            )


            comments = get_data(
                "weui-desktop-mass-media__data appmsg-comment"
            )



            page_data.append(
                {
                    "标题":title,
                    "阅读人数":read_count,
                    "点赞人数":likes,
                    "分享人数":shares,
                    "在看人数":on_view,
                    "留言条数":comments
                }
            )


        except Exception as e:

            print(
                "文章解析失败:",
                e
            )


    print(
        f"本页提取 {len(page_data)} 篇文章"
    )


    return page_data




# =========================
# 自动翻页
# =========================

all_articles_data=[]

page=1



while len(all_articles_data)<100:


    print(
        f"正在提取第{page}页"
    )


    current_data = extract_page_data()


    all_articles_data.extend(
        current_data
    )



    if len(all_articles_data)>=100:

        break



    try:


        next_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.LINK_TEXT,
                    "下一页"
                )
            )
        )


        next_button.click()


        page+=1


        print(
            "点击下一页"
        )


        time.sleep(1)



    except Exception as e:


        print(
            "没有下一页:",
            e
        )

        break



# 关闭浏览器

driver.quit()



# =========================
# 保存Excel
# =========================

all_articles_data = all_articles_data[:50]


if all_articles_data:


    df=pd.DataFrame(
        all_articles_data
    )


    output_file=os.path.join(
        os.path.expanduser("~"),
        "Desktop",
        "微信文章数据.xlsx"
    )



    df.to_excel(
        output_file,
        index=False,
        sheet_name="微信文章数据"
    )



    print(
        f"保存成功:{output_file}"
    )



    os.startfile(
        output_file
    )


else:


    print(
        "没有提取到数据"
    )

请登录微信公众号...
登录成功！
进入发表记录页面
正在提取第1页
本页提取 13 篇文章
点击下一页
正在提取第2页
本页提取 11 篇文章
点击下一页
正在提取第3页
本页提取 12 篇文章
点击下一页
正在提取第4页
本页提取 10 篇文章
点击下一页
正在提取第5页
本页提取 10 篇文章
点击下一页
正在提取第6页


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=151.0.7872.0); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff70ed12295+14fa5]
	chromedriver!GetHandleVerifier [0x7ff70ed122f0+15000]
	chromedriver!(No symbol) [0x7ff70e845bad]
	chromedriver!(No symbol) [0x7ff70e831372]
	chromedriver!(No symbol) [0x7ff70e8572ed]
	chromedriver!(No symbol) [0x7ff70e8d0b10]
	chromedriver!(No symbol) [0x7ff70e8eda42]
	chromedriver!(No symbol) [0x7ff70e892f9c]
	chromedriver!(No symbol) [0x7ff70e893ec3]
	chromedriver!GetHandleVerifier [0x7ff70f2effb1+5f2cc1]
	chromedriver!GetHandleVerifier [0x7ff70f2ea2ea+5ecffa]
	chromedriver!GetHandleVerifier [0x7ff70f30df55+610c65]
	chromedriver!GetHandleVerifier [0x7ff70ed2ffbe+32cce]
	chromedrive